# VGG by PyTorch

Source : https://pytorch.org/hub/pytorch_vision_vgg/ dont les performances Top-1 et Top-5.

>Notes : En plus du changement `wget` -> `curl`, il faudra adapter le code  pour un mac (`mps`au lieu de `cuda`).

In [1]:
import torch
#model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg11', pretrained=True)
# or any of these variants
#model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg11_bn', pretrained=True)
#model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg13', pretrained=True)
#model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg13_bn', pretrained=True)
model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg16', pretrained=True)
#model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg16_bn', pretrained=True)
#model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg19', pretrained=True)
#model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg19_bn', pretrained=True)
model.eval()

# bn : batchnorm

Using cache found in /Users/me/.cache/torch/hub/pytorch_vision_v0.10.0
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /Users/me/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|████████████████████████████████████████████████████████████████████████████████| 528M/528M [00:08<00:00, 68.7MB/s]


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [2]:
import urllib
url, filename = ("https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")
try: urllib.URLopener().retrieve(url, filename)
except: urllib.request.urlretrieve(url, filename)

In [3]:
# sample execution (requires torchvision)
from PIL import Image
from torchvision import transforms
input_image = Image.open(filename)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess(input_image)
input_batch = input_tensor.unsqueeze(0) # create a mini-batch as expected by the model

# move the input and model to GPU for speed if available
if torch.cuda.is_available():
    input_batch = input_batch.to('cuda')
    model.to('cuda')

with torch.no_grad():
    output = model(input_batch)
# Tensor of shape 1000, with confidence scores over ImageNet's 1000 classes
print(output[0])
# The output has unnormalized scores. To get probabilities, you can run a softmax on it.
probabilities = torch.nn.functional.softmax(output[0], dim=0)
print(probabilities)

tensor([-2.3516e+00, -2.5588e+00, -1.4561e+00, -2.9489e+00, -2.5636e+00,
        -1.9037e+00, -2.7426e+00,  2.0772e+00,  4.2542e+00, -1.4162e+00,
        -3.7596e+00, -2.4730e+00, -3.2577e+00, -2.0014e+00, -2.4202e+00,
        -2.5381e+00, -1.0395e+00, -5.4885e-01,  2.9824e-01, -2.4372e+00,
        -1.9157e+00, -1.5719e+00, -6.8911e-01,  9.3941e-01, -2.2614e+00,
        -2.4488e+00, -3.1065e+00, -1.4743e+00, -2.2206e+00, -8.2710e-01,
        -3.6595e+00, -2.4025e+00, -3.2567e+00, -4.3827e+00, -3.3335e+00,
        -3.3722e+00, -2.2806e+00, -2.4148e+00, -4.3653e+00, -2.8325e+00,
        -1.7348e+00, -3.4131e+00, -4.6667e+00, -4.0632e+00, -3.6034e+00,
        -2.4997e+00, -3.9418e-02, -3.7291e+00, -3.4300e+00, -2.5797e+00,
        -1.2160e+00, -3.4979e+00, -1.5980e+00, -2.9979e+00, -2.7307e+00,
        -2.1657e+00, -3.0534e+00, -3.4219e+00, -2.9683e+00, -2.5575e+00,
        -1.6242e+00, -4.8267e+00, -4.6957e+00, -2.9213e+00, -2.8225e+00,
        -4.0444e+00, -3.6463e+00, -3.3712e+00, -4.5

In [4]:
# Download ImageNet labels
#!wget https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt
!curl -O https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10472  100 10472    0     0   8052      0  0:00:01  0:00:01 --:--:--  8049      0      0 --:--:-- --:--:-- --:--:--     0


In [5]:
# Read the categories
with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]
# Show top categories per image
top5_prob, top5_catid = torch.topk(probabilities, 5)
for i in range(top5_prob.size(0)):
    print(categories[top5_catid[i]], top5_prob[i].item())

Samoyed 0.8395193815231323
Pomeranian 0.0329851359128952
Eskimo dog 0.01683262549340725
white wolf 0.01595180109143257
Great Pyrenees 0.0121866250410676
